# Audit dataset kredit: tahap 1-3

**Aqil — L0124139.** Menjaga raw, memeriksa data, menyusun kamus variabel, dan menetapkan aturan. Tidak membuat split/model. Script `audit_dataset.py` harus tetap berada di samping notebook. Dependensi: pandas dan numpy.

Sel notebook yang tersimpan sudah dijalankan berurutan memakai Python; lingkungan pembuat tidak menyediakan kernel Jupyter.

## 1. Temukan modul audit

Dapat dijalankan dari folder `kredit` atau root proyek. Pembacaan dan penulisan hasil audit dilakukan oleh fungsi di script yang dapat diperiksa.

In [1]:
import importlib.util
from pathlib import Path

base = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'audit_dataset.py').is_file()), None)
if base is None:
    base = next((p / 'kredit' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'kredit' / 'audit_dataset.py').is_file()), None)
if base is None:
    raise FileNotFoundError('Jalankan dari root proyek atau folder kredit.')
spec = importlib.util.spec_from_file_location('credit_audit', base / 'audit_dataset.py')
audit_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit_module)
print('Modul audit ditemukan.')


Modul audit ditemukan.


## 2. Simpan snapshot raw dan manifest

Sumber asli tidak ditimpa. Snapshot yang sudah ada harus memiliki checksum sama; jika berbeda, proses berhenti. Tanggal unduh dan versi Kaggle tidak diketahui, sehingga dicatat kosong.

In [2]:
ctx = audit_module.prepare_source()
print('Dimensi:', ctx['df'].shape)
print('SHA-256:', ctx['hash'])
print('Raw snapshot dan sumber asli identik:', audit_module.digest(ctx['snapshot']) == ctx['hash'])


Dimensi: (30000, 25)
SHA-256: a0f0ab49d6326671d6cd83be5c88dcf18007025fe9a53ecd699119c871176ca1
Raw snapshot dan sumber asli identik: True


## 3. Audit kualitas dan variasi

Ini audit seluruh data. Belum ada threshold outlier, seleksi fitur, atau transformasi yang dipelajari. Pengulangan berdasarkan fitur tidak otomatis berarti orang yang sama.

In [3]:
summary = audit_module.audit(ctx)
for key in ['missing_cells','nonfinite_values','n_ID_unik','duplicate_excluding_ID_excess','rows_in_repeated_feature_profiles','conflicting_label_groups','education_0_5_6_rows','marriage_0_rows','rows_with_negative_bill']:
    print(key, ':', summary[key])
print('\nDistribusi target:')
print(ctx['distribution'].to_string(index=False))
print('\nProfil 25 kolom:')
print(ctx['profile'].to_string(index=False))


missing_cells : 0
nonfinite_values : 0
n_ID_unik : 30000
duplicate_excluding_ID_excess : 35
rows_in_repeated_feature_profiles : 108
conflicting_label_groups : 21
education_0_5_6_rows : 345
marriage_0_rows : 54
rows_with_negative_bill : 1930

Distribusi target:
 kode  jumlah  persen              arti
    0   23364   77.88 Tidak gagal bayar
    1    6636   22.12       Gagal bayar

Profil 25 kolom:
                     kolom   dtype  jumlah_kosong  nilai_unik   minimum   median  maksimum  jumlah_nol  jumlah_negatif
                        ID   int64              0       30000       1.0  15000.5   30000.0           0               0
                 LIMIT_BAL float64              0          81   10000.0 140000.0 1000000.0           0               0
                       SEX   int64              0           2       1.0      2.0       2.0           0               0
                 EDUCATION   int64              0           7       0.0      2.0       6.0          14               0
      

## 4. Kamus variabel

S1=UCI; S2=mirror Kaggle; S3=Lin dan Wang (preprint 2025). Arti PAY -2/0 dibedakan sumbernya dari definisi utama UCI. Kategori unknown adalah kebijakan konservatif kelompok.

In [4]:
data_dictionary = audit_module.dictionary(ctx)
print(data_dictionary[['kolom','arti','jenis_semantik','satuan']].to_string(index=False))
print('\nDokumen lengkap: data_dictionary.md')


                     kolom                                arti jenis_semantik        satuan
                        ID                   Identitas nasabah             ID Tidak berlaku
                 LIMIT_BAL         Batas kredit yang diberikan        Numerik     NT dollar
                       SEX Kategori jenis kelamin yang direkam      Kategoris          Kode
                 EDUCATION                 Kategori pendidikan      Kategoris          Kode
                  MARRIAGE          Kategori status perkawinan      Kategoris          Kode
                       AGE                        Usia nasabah        Numerik         Tahun
                     PAY_0    Status pembayaran September 2005      Kategoris          Kode
                     PAY_2      Status pembayaran Agustus 2005      Kategoris          Kode
                     PAY_3         Status pembayaran Juli 2005      Kategoris          Kode
                     PAY_4         Status pembayaran Juni 2005      Kategoris   

## 5. Keputusan pembersihan

Aturan didokumentasikan tanpa menerapkan pembersihan ke raw. Pemisahan target/ID, pemetaan kategori, dan encoder dijalankan pada tahap berikutnya. Nilai negatif, nol, label berbeda dan fitur berkorelasi tidak dihapus otomatis.

In [5]:
rules = audit_module.cleaning_rules(ctx)
print(rules[['kode','isu','temuan','status']].to_string(index=False))
print('\nDokumen lengkap: data_cleaning_decisions.md')


kode                             isu                                               temuan                                                    status
 P01                     Data mentah                               30,000 baris, 25 kolom                     Selesai: snapshot dan manifest dibuat
 P02                        Kolom ID                                       30,000 ID unik           Ditetapkan; diterapkan pada tahap persiapan X/y
 P03                          Target           Kode 0/1; gagal bayar 22.12% pada file ini                           Ditetapkan; belum membuat split
 P04               Missing/nonfinite                            0 sel kosong; 0 nonfinite                           Ditetapkan; tidak ada pengisian
 P05                 EDUCATION 0/5/6                                            345 baris       Ditetapkan untuk tahap berikutnya; raw belum diubah
 P06                      MARRIAGE 0                                             54 baris               Ditetapk

## 6. Verifikasi dan serah terima

Cek sumber dan raw tetap sama, data yang dibaca tidak dimutasi, kamus mencakup seluruh kolom, dan hitungan saling sesuai. Output utama Aqil: README, kamus, dan keputusan pembersihan. Hasil audit bukan data siap model.

In [6]:
checks = audit_module.verify_and_report(ctx)
for name, passed in checks.items():
    print(name, ':', passed)
print('\nSelesai tahap 1-3. Tidak membuat split, encoding, model atau TreeSHAP.')


original_unchanged : True
raw_snapshot_matches_original : True
raw_dataframe_not_modified : True
dictionary_covers_each_column_once : True
target_counts_reconcile : True
category_counts_reconcile : True
duplicate_members_reconcile : True
target_is_binary_and_complete : True

Selesai tahap 1-3. Tidak membuat split, encoding, model atau TreeSHAP.
